In [ ]:
#! python -m pip install numpy scipy matplotlib
%load_ext autoreload
%autoreload 2

import socket
import time
import numpy as np
import scipy.signal as signal
from collections import deque
from IPython.display import clear_output

# --- CONFIGURACIÓN ---
UDP_IP = "0.0.0.0"
UDP_PORT = 5005
SUBPORTADORAS_OBJETIVO = 74
TAMAÑO_VENTANA = 500
INDICE_RENOVACION = 20

def parse_csi_payload(data):
    csi_raw = np.frombuffer(data, dtype=np.int8)
    real = csi_raw[0::2].astype(np.float32)
    imag = csi_raw[1::2].astype(np.float32)
    return real + 1j * imag

# Inicializar socket UDP con opción de reutilizar dirección
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
sock.bind((UDP_IP, UDP_PORT))

print(f"Escuchando datos CSI en el puerto {UDP_PORT} (UDP)...")

csi_buffer = deque(maxlen=TAMAÑO_VENTANA)
paquetes = 0

try:
    while True:
        # 1. RECEPCIÓN DE DATOS EN TIEMPO REAL
        data, addr = sock.recvfrom(4096)
        if len(data) < 10:
            continue
            
        csi_complex = parse_csi_payload(data)
        
        # Filtrar paquetes para garantizar matriz homogénea
        if len(csi_complex) != SUBPORTADORAS_OBJETIVO:
            continue
            
        # Guardar en el buffer dinámico
        csi_buffer.append(csi_complex)
        paquetes += 1

        # 2. ESPERAR A LLENAR LA VENTANA INICIAL DE 500
        if len(csi_buffer) < TAMAÑO_VENTANA:
            if paquetes % 20 == 0:
                clear_output(wait=True)
                print(f"Llenando buffer inicial... {len(csi_buffer)}/{TAMAÑO_VENTANA} paquetes.")
            continue

        # 3. PROCESAMIENTO CADA N PAQUETES RECIBIDOS
        if paquetes % INDICE_RENOVACION == 0:
            csi_largo = np.array(csi_buffer) # Matriz de (500, 74)

            fases_brutas = np.angle(csi_largo)
            fases_unwrapped = np.unwrap(fases_brutas, axis=0)

            # Filtro Butterworth
            fs = 100  # Frecuencia estimada (paquetes/s)
            nyquist = fs / 2
            low, high = 0.1 / nyquist, 0.5 / nyquist
            b, a = signal.butter(N=2, Wn=[low, high], btype="bandpass")
            fases_filtradas = signal.filtfilt(b, a, fases_unwrapped, axis=0)

            # Selección de subportadora dominante
            varianzas = np.var(fases_filtradas, axis=0)
            mejor_subportadora = np.argmax(varianzas)
            mejor_senal = fases_filtradas[:, mejor_subportadora]

            # FFT para estimar RPM
            n_fft = 10000
            fft_valores = np.fft.fft(mejor_senal, n=n_fft)
            magnitudes = np.abs(fft_valores[: n_fft // 2])
            frecuencias = np.fft.fftfreq(n_fft, d=1 / fs)[: n_fft // 2]
            rpm_pos = frecuencias * 60.0

            margen_humana = (rpm_pos >= 12.0) & (rpm_pos <= 30.0)
            rpm_validas = rpm_pos[margen_humana]
            magnitudes_validas = magnitudes[margen_humana]

            # 4. SALIDA DINÁMICA EN CONSOLA
            clear_output(wait=True)
            print(f"--- MONITOREO EN TIEMPO REAL ---")
            print(f"Paquetes procesados: {paquetes}")
            
            if len(magnitudes_validas) > 0:
                indice_pico = np.argmax(magnitudes_validas)
                pico_potencia = magnitudes_validas[indice_pico]
                promedio_ruido = np.mean(magnitudes)

                if pico_potencia > (3.0 * promedio_ruido):
                    rpm = rpm_validas[indice_pico]
                    print(f"ESTADO: PRESENCIA DETECTADA")
                    print(f"RESPIRACIÓN: {rpm:.1f} RPM (Subportadora {mejor_subportadora})")
                else:
                    print("ESTADO: SIN PRESENCIA DETECTADA (Señal en reposo)")

except KeyboardInterrupt:
    print("\nEjecución detenida por el usuario.")
finally:
    sock.close()

IndentationError: unexpected indent (589631767.py, line 6)